# Fase 2 — Transfer Learning mejorado (MobileNetV2 / MobileNetV3Small)

Este notebook está pensado para la **segunda fase del challenge**.

## Objetivos
1. Cargar un dataset real desde carpetas.
2. Comparar un modelo **from scratch** con un modelo **preentrenado**.
3. Probar dos variantes:
   - **Feature extractor congelado**
   - **Fine-tuning parcial**
4. Analizar:
   - accuracy
   - pérdida
   - overfitting
   - tiempo aproximado de entrenamiento

## Estructura esperada del dataset

```text
dataset_chihuahua_muffin/
    train/
        chihuahua/
        muffin/
    val/
        chihuahua/
        muffin/
    test/
        chihuahua/
        muffin/
```


## 0. Identificación del equipo

Rellenad esto al empezar.


In [ ]:
TEAM_NAME = "Equipo ..."
TEAM_MEMBERS = [
    "Nombre Apellido",
    "Nombre Apellido",
]
MODEL_BASELINE = "CNN from scratch"
TRANSFER_MODEL = "MobileNetV2"   # o MobileNetV3Small


## 1. Imports

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_v2
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_v3

print("TensorFlow:", tf.__version__)


## 2. Configuración

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = Path("dataset_chihuahua_muffin")  # cambiar si hace falta
USE_MOBILENET = "v2"  # "v2" o "v3"

assert DATA_DIR.exists(), f"No existe {DATA_DIR}"


## 3. Carga del dataset desde carpetas

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print("Clases:", class_names)


## 4. Visualización rápida

In [ ]:
plt.figure(figsize=(8,8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Optimización del pipeline

Usamos `cache`, `shuffle` y `prefetch` para acelerar el entrenamiento.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds_opt = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds_opt = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_opt = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 6. Data augmentation

Esto se aplicará **solo en entrenamiento**.

Prueba a activar o desactivar capas si quieres experimentar.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")


## 7. Baseline from scratch

Primero entrenamos una CNN propia para compararla luego con MobileNet.


In [ ]:
def build_scratch_model():
    model = tf.keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        layers.Rescaling(1./255),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

scratch_model = build_scratch_model()
scratch_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
scratch_model.summary()


In [ ]:
callbacks_common = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
]

start = time.time()
scratch_history = scratch_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=8,
    callbacks=callbacks_common,
    verbose=1
)
scratch_time = time.time() - start
print(f"Tiempo entrenamiento scratch: {scratch_time:.1f} s")


In [ ]:
scratch_test_loss, scratch_test_acc = scratch_model.evaluate(test_ds_opt, verbose=0)
print(f"Scratch test acc: {scratch_test_acc:.4f}")


## 8. Funciones auxiliares para gráficas

In [ ]:
def plot_history(history, title="Modelo"):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.title(f"Loss — {title}")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="val")
    plt.title(f"Accuracy — {title}")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(scratch_history, "CNN from scratch")


## 9. Transfer learning — seleccionar MobileNet

Usamos:
- `preprocess_v2` para MobileNetV2
- `preprocess_v3` para MobileNetV3Small

Esto es importante: **no basta con reescalar**, hay que usar el preprocesado adecuado del modelo.


In [ ]:
if USE_MOBILENET == "v2":
    BaseClass = MobileNetV2
    preprocess_fn = preprocess_v2
    chosen_name = "MobileNetV2"
else:
    BaseClass = MobileNetV3Small
    preprocess_fn = preprocess_v3
    chosen_name = "MobileNetV3Small"

print("Modelo elegido:", chosen_name)


## 10. Modelo transfer learning (feature extractor congelado)


In [ ]:
def build_transfer_model(base_class, preprocess_fn, trainable_base=False):
    base_model = base_class(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = trainable_base

    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = preprocess_fn(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

transfer_model, base_model = build_transfer_model(BaseClass, preprocess_fn, trainable_base=False)

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
transfer_model.summary()


In [ ]:
start = time.time()
transfer_history = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=8,
    callbacks=callbacks_common,
    verbose=1
)
transfer_time = time.time() - start
print(f"Tiempo entrenamiento transfer congelado: {transfer_time:.1f} s")


In [ ]:
transfer_test_loss, transfer_test_acc = transfer_model.evaluate(test_ds_opt, verbose=0)
print(f"{chosen_name} congelado — test acc: {transfer_test_acc:.4f}")
plot_history(transfer_history, f"{chosen_name} congelado")


## 11. Fine-tuning parcial

Descongelamos solo las últimas capas y bajamos el learning rate.


In [ ]:
base_model.trainable = True

# Congelar casi todas y dejar entrenables las últimas 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

start = time.time()
finetune_history = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=5,
    callbacks=callbacks_common,
    verbose=1
)
finetune_time = time.time() - start
print(f"Tiempo fine-tuning: {finetune_time:.1f} s")


In [ ]:
finetune_test_loss, finetune_test_acc = transfer_model.evaluate(test_ds_opt, verbose=0)
print(f"{chosen_name} fine-tuned — test acc: {finetune_test_acc:.4f}")
plot_history(finetune_history, f"{chosen_name} fine-tuned")


## 12. Comparación final


In [ ]:
print("="*60)
print("COMPARACIÓN FINAL")
print("="*60)
print(f"Scratch CNN            -> acc={scratch_test_acc:.4f} | tiempo={scratch_time:.1f}s")
print(f"{chosen_name} congelado -> acc={transfer_test_acc:.4f} | tiempo={transfer_time:.1f}s")
print(f"{chosen_name} fine-tune -> acc={finetune_test_acc:.4f} | tiempo={finetune_time:.1f}s")


## 13. Preguntas obligatorias para el equipo

Responded en una celda Markdown o en el documento de entrega:

1. ¿Qué modelo generaliza mejor?
2. ¿Cuál entrena más rápido?
3. ¿Compensa usar transfer learning en este dataset?
4. ¿Ha mejorado el fine-tuning respecto al modelo congelado?
5. Si cambiarais a un dataset más difícil, ¿qué enfoque usaríais?


## 14. Resultado para leaderboard

Rellenad esto al final.


In [ ]:
FINAL_MODEL = chosen_name
FINAL_TEST_ACCURACY = float(finetune_test_acc)  # o la mejor que hayáis obtenido
FINAL_NOTES = "Scratch vs transfer comparado"
print(TEAM_NAME, FINAL_MODEL, FINAL_TEST_ACCURACY, FINAL_NOTES)
